# NGHIÊN CỨU VÀ TRỰC QUAN HÓA CÁC ĐẶC TRƯNG ÂM THANH TRÊN SPOTIFY

In [20]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'

df_tracks = pd.read_csv(f'{data_dir}/tracks_dimension.csv')
df_country_year = pd.read_parquet(f'{data_dir}/mart_country_year.parquet')
df_weekly = pd.read_parquet(f'{data_dir}/mart_audio_trend_weekly.parquet')
df_artist = pd.read_parquet(f'{data_dir}/mart_artist_chart.parquet')


## Ma trận tương quan tuyến tính giữa các đặc trưng âm học

In [21]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
df_charts = pd.read_parquet(f'{data_dir}/charts_clean.parquet')

all_features4 = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'loudness', 'tempo']
labels4 = ['Danceability', 'Energy', 'Valence', 'Acousticness', 'Speechiness', 'Instrumentalness', 'Liveness', 'Loudness', 'Tempo']

df_global4 = df_charts[
    (df_charts['chart_country'] == 'Global') &
    (df_charts['year'].between(2020, 2022))
].copy()

df_global4 = df_global4.drop_duplicates(
    subset=['chart_country', 'week', 'rank', 'track_id']
)

band_order4 = [
    'Top 1 đến 10',
    'Hạng 11 đến 50',
    'Hạng 51 đến 100',
    'Hạng 101 đến 200'
]

df_global4['rank_band'] = pd.cut(
    df_global4['rank'],
    bins=[0, 10, 50, 100, 200],
    labels=band_order4,
    include_lowest=True
)

mean_table4 = (
    df_global4
    .groupby('rank_band', observed=False)[all_features4]
    .mean()
    .loc[band_order4]
)

overall_mean4 = df_global4[all_features4].mean()
overall_std4 = df_global4[all_features4].std(ddof=0).replace(0, np.nan)
z_table4 = (mean_table4 - overall_mean4) / overall_std4

text_z4 = [[f'{val:+.2f}' for val in row] for row in z_table4.values]

customdata4 = np.dstack([
    mean_table4.values,
    z_table4.values
])

max_abs4 = float(np.nanmax(np.abs(z_table4.values)))

fig4 = go.Figure(
    data=go.Heatmap(
        z=z_table4.values,
        x=labels4,
        y=band_order4,
        text=text_z4,
        texttemplate='%{text}',
        textfont=dict(size=14),
        customdata=customdata4,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-max_abs4,
        zmax=max_abs4,
        colorbar=dict(
            title='Độ lệch<br>chuẩn z-score',
            thickness=16,
            len=0.85
        ),
        hovertemplate=(
            '<b>Nhóm:</b> %{y}<br>'
            '<b>Đặc trưng:</b> %{x}<br>'
            '<b>Trung bình thực:</b> %{customdata[0]:.3f}<br>'
            '<b>Z-score:</b> %{customdata[1]:+.2f} SD'
            '<extra></extra>'
        )
    )
)

fig4.update_layout(
    title=dict(
        text='<b>Hồ Sơ Âm Thanh Theo Phân Tầng Thứ Hạng</b>',
        x=0.01,
        font=dict(size=16)
    ),
    xaxis=dict(tickangle=-25, tickfont=dict(size=12)),
    yaxis=dict(autorange='reversed', tickfont=dict(size=12)),
    height=480,
    margin=dict(l=120, r=40, t=75, b=80),
    template='plotly_white'
)
fig4.show()


### Cách tính

$$
z =
\frac{
\text{Trung bình nhóm thứ hạng}
-
\text{Trung bình toàn BXH}
}{
\text{Độ lệch chuẩn toàn BXH}
}
$$
- $z>0$: cao hơn trung bình.
- $z<0$: thấp hơn trung bình.
- $|z|$ càng lớn: khác biệt càng rõ.

### Hiện tượng

- **Top 1–10 nổi bật với sắc đỏ dương tính:** Vượt trội so với mức trung bình toàn bảng xếp hạng ở **Tempo (+0.16 SD)**, **Liveness (+0.14 SD)**, **Danceability (+0.13 SD)**, **Valence (+0.12 SD)** và **Energy (+0.11 SD)**.
- **Sắc xanh âm tính ở nhóm nhạc mộc và không lời:** Top 1–10 thấp hơn trung bình ở **Instrumentalness (-0.12 SD)**, **Acousticness (-0.11 SD)** và **Speechiness (-0.11 SD)**.
- **Loudness gần như bất biến (-0.00 SD):** Màu sắc trung tính thể hiện âm lượng không tạo ra sự khác biệt giữa các phân tầng.

So sánh trực tiếp giữa **Top 1–10** và **Hạng 101–200**:
- **Tempo:** Nhanh hơn **5,53 BPM** (125,78 BPM so với 120,24 BPM).
- **Valence:** Tươi vui hơn **0,035** (0,546 so với 0,511).
- **Danceability:** Dễ nhảy hơn **0,026** (0,704 so với 0,678).
- **Energy:** Sôi động hơn **0,022** (0,647 so với 0,625).
- **Acousticness:** Thấp hơn **0,037** (0,232 so với 0,269).
- **Speechiness:** Thấp hơn **0,012** (0,096 so với 0,108) và **Instrumentalness** thấp hơn **0,010** (0,001 so với 0,011).
- **Loudness:** Chênh lệch thực tế chỉ **0,068 dB** (-6,18 dB so với -6,24 dB).

### Cơ chế giải thích

- **Âm nhạc đại chúng ưu tiên nhịp điệu và năng lượng:** Các ca khúc Top 10 có xu hướng nhanh, bắt tai, tích cực và giàu tính kích thích chuyển động (Danceability, Energy, Tempo cao) nhằm tối ưu cho các ngữ cảnh nghe phổ biến như playlist giải trí, tập luyện, video ngắn và không gian công cộng.
- **Sự thống trị của giọng hát và sản xuất điện tử:** Chỉ số Instrumentalness (-0.12 SD) và Acousticness (-0.11 SD) rất thấp phản ánh xu hướng các bài hit luôn dựa vào giọng hát nổi bật (vocal hooks) và bản phối kỹ thuật số hiện đại.
- **Chỉ số Speechiness (-0.11 SD) thấp:** Không đồng nghĩa với việc Rap kém phổ biến, mà do các bản rap lọt đỉnh bảng xếp hạng thường được melodic hóa (kết hợp giai điệu Pop/R&B dễ hát theo), làm giảm mật độ lời nói thuần túy so với các bản rap underground ở nhóm dưới.

### Kết luận

 Các bài hát Top 10 sở hữu một “dấu vân tay âm học” riêng biệt: nhanh hơn, dễ nhảy hơn, tích cực hơn và mang tính thương mại cao hơn nhóm cuối bảng. Tuy nhiên, mức chênh lệch z-score (+0.11 đến +0.16 SD) là tương đối vừa phải, cho thấy đặc trưng âm thanh tạo lợi thế cạnh tranh nhưng không phải là “công thức độc tôn” quyết định vị trí quán quân.


## Phân phối đặc trưng âm học giữa các thể loại nhạc

In [22]:
main_genres = ['pop', 'latin', 'k-pop', 'rap', 'dance pop']
df_genre_filtered = df_tracks[df_tracks['artist_genre'].isin(main_genres)]

fig7 = go.Figure()
box_metrics = ['energy', 'danceability', 'valence', 'tempo']
box_titles = ['Energy', 'Danceability', 'Valence', 'Tempo']

for i, m in enumerate(box_metrics):
    for g in main_genres:
        sub = df_genre_filtered[df_genre_filtered['artist_genre'] == g]
        fig7.add_trace(go.Box(
            y=sub[m],
            name=g,
            visible=(i == 0)
        ))

buttons7 = []
n_g = len(main_genres)
for i, name in enumerate(box_titles):
    vis = [False] * (len(box_metrics) * n_g)
    for j in range(n_g):
        vis[i * n_g + j] = True
    buttons7.append(dict(
        label=name,
        method='update',
        args=[{'visible': vis}, {'title': 'Phân phối chỉ số ' + name + ' giữa các thể loại âm nhạc'}]
    ))

fig7.update_layout(
    title='Phân phối chỉ số Energy giữa các thể loại âm nhạc',
    updatemenus=[dict(buttons=buttons7, direction='down', x=0.01, y=1.15, showactive=True)],
    height=550
)
fig7.show()


### Nhận xét Biểu đồ 
**Hiện tượng:** Biểu đồ tương tác cho thấy bản sắc âm học phân hóa rõ rệt giữa 5 dòng nhạc khi chuyển đổi qua 4 chiều đặc trưng:
- **Energy (Năng lượng):** K-Pop (trung vị 0.73) và Latin (0.72) áp đảo với dải phân vị tập trung ở mức cao; Pop (0.60) và Rap (0.63) có mức năng lượng thấp hơn và độ phân tán rộng hơn.
- **Danceability (Độ dễ nhảy):** Khi chuyển sang Danceability, Latin (0.76) và Rap (0.75) vươn lên dẫn đầu toàn diện với dải hộp co hẹp ở mức rất cao, trong khi Pop (0.65) và K-Pop (0.66) khiêm tốn hơn.
- **Valence (Sắc thái cảm xúc):** Latin độc chiếm vị trí tươi vui nhất (trung vị 0.67, bỏ xa các dòng nhạc khác gần 0.20 đơn vị), thể hiện bản sắc lễ hội đặc trưng; ngược lại Rap (0.44) và Pop (0.45) có phổ cảm xúc thiên về gam màu trầm và u tối hơn.
- **Tempo (Tốc độ nhịp BPM):** Rap dẫn đầu tốc độ với trung vị 126.8 BPM (đặc trưng của beat Trap nhanh); trong khi Latin giữ nhịp đầm chắc nhất quanh 111.0 BPM (nhịp Dembow/Reggaeton).

**Cơ chế giải thích:** 
- Sự khác biệt về Energy và Danceability bắt nguồn từ mục đích biểu diễn: Dance Pop và K-Pop phục vụ trình diễn sân khấu vũ đạo lớn với mật độ âm thanh dày, còn Latin và Rap xây dựng trên nền tảng nhịp groove (Dembow và Hip-hop beat) kích thích chuyển động hình thể tự nhiên.
- Sự phân cực về Valence phản ánh văn hóa âm nhạc: dòng nhạc Latin mang đậm tinh thần tiệc tùng và tích cực, trong khi Rap và Pop chứa đựng nhiều ca từ tự sự, nội tâm và tâm trạng đời sống.

**Kết luận:** Không có thể loại nào "cao hơn toàn diện" trên mọi phương diện; mỗi dòng nhạc đều có một thế mạnh đặc thù (Latin mạnh về Danceability & Valence, K-Pop mạnh về Energy, Rap mạnh về Tempo). Bộ lọc tương tác chứng minh sự cần thiết của việc đánh giá âm nhạc đa chiều thay vì chỉ nhìn vào một chỉ số đơn lẻ.


## Phân cấp thị phần thể loại và nghệ sĩ dẫn đầu

In [23]:
top_artists_genre = df_artist[df_artist['artist_genre'].isin(['pop', 'latin', 'k-pop', 'rap', 'canadian pop', 'dance pop', 'trap latino'])].nlargest(80, 'global_streams')

fig9 = px.treemap(
    top_artists_genre,
    path=['artist_genre', 'artist_name'],
    values='global_streams',
    title='hân cấp thị phần thể loại và nghệ sĩ dẫn đầu',
    labels={'global_streams': 'Tổng lượt streams', 'artist_genre': 'Thể loại', 'artist_name': 'Nghệ sĩ'},
    color_continuous_scale='Purples'
)
fig9.update_layout(height=550)
fig9.show()


### Nhận xét Biểu đồ

**Hiện tượng:**

- Pop độc chiếm diện tích lớn nhất với 36 nghệ sĩ góp mặt. Các tên tuổi dẫn đầu nhánh này gồm Ed Sheeran, Justin Bieber, The Weeknd và Billie Eilish.
- Âm nhạc Latinh tạo thành cực đối trọng khổng lồ: Khi kết hợp cả hai nhánh trap latino và latin, dòng nhạc Latinh chiếm tới 26,6% thị phần (hơn 94,2 tỷ streams). Đặc biệt, Bad Bunny sở hữu ô chữ nhật đơn lẻ lớn nhất toàn biểu đồ, theo sau là J Balvin 
- Dance Pop và Canadian Pop khẳng định sức mạnh đại chúng: dance pop với các đại diện tiêu biểu như Dua Lipa và Ariana Grande; trong khi nhánh canadian pop tuy chỉ có 1 đại diện độc lập nhưng Drake chiếm riêng tới 4,7% (16,7 tỷ streams).
- Rap và K-Pop chiếm diện tích khiêm tốn hơn trong nhóm 80 nghệ sĩ này.

**Cơ chế giải thích:**

- Trên các nền tảng streaming trực tuyến, thuật toán đề xuất và văn hóa nghe nhạc toàn cầu có xu hướng khuếch đại lượt nghe cho các nghệ sĩ Pop và Latinh hàng đầu, tạo nên các ô diện tích áp đảo.
- Tính dễ tiếp cận và khả năng lan tỏa xuyên biên giới: Pop và Latinh sở hữu các đặc trưng âm học bắt tai (Danceability và Energy cao, nhịp điệu dễ ghi nhớ), giúp các bài hát dễ dàng vượt qua rào cản ngôn ngữ để thống trị các bảng xếp hạng đa quốc gia.

**Kết luận:** 

Thị trường âm nhạc trực tuyến có tính phân cấp thị phần và tích tụ lưu lượng rõ rệt: Pop và Latinh cùng các siêu sao đầu ngành (Bad Bunny, Drake, Ed Sheeran) đóng vai trò đầu tàu chi phối phần lớn thời lượng nghe trên Spotify toàn cầu.

## Tính mùa vụ trong âm nhạc theo 12 tháng


In [24]:
df_glob = df_weekly[df_weekly['chart_country'] == 'Global'].copy()
df_glob['month'] = pd.to_datetime(df_glob['week']).dt.month
monthly_avg = df_glob.groupby('month')[['danceability_weighted', 'acousticness_weighted']].mean().reset_index()

fig10 = px.bar(
    monthly_avg,
    x='month',
    y=['danceability_weighted', 'acousticness_weighted'],
    barmode='group',
    title='Tính mùa vụ trong âm nhạc: Danceability vs Acousticness theo 12 tháng',
    labels={'month': 'Tháng trong năm', 'value': 'Giá trị trung bình', 'variable': 'Đặc trưng âm học'},
    color_discrete_map={'danceability_weighted': '#1DB954', 'acousticness_weighted': '#E67E22'}
)
fig10.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    height=500
)
fig10.show()


### Nhận xét biểu đồ

Chỉ số Danceability duy trì ổn định quanh mức 0.70 từ tháng 1 đến tháng 11, nhưng suy giảm rõ rệt vào tháng 12. Ngược lại, Acousticness duy trì ở mức thấp khoảng 0.22 đến 0.24 trong suốt 11 tháng đầu năm, sau đó tăng vọt lên mức cao nhất vào tháng 12.

**Nguyên nhân:** Hiện tượng biến thiên đột ngột vào tháng 12 là hệ quả trực tiếp của Giáng sinh. Trong giai đoạn này, các bài hát kinh điển truyền thống từ thế kỷ trước vốn sử dụng nhạc cụ mộc, dàn dây và ít phách điện tử tái xuất hiện và chiếm lĩnh các vị trí dẫn đầu bảng xếp hạng, làm thay đổi cấu trúc âm học bình quân của toàn bộ thị trường.

**Kết luận:** Âm nhạc trực tuyến sở hữu tính mùa vụ chu kỳ rõ rệt hàng năm, trong đó tháng 12 là giai đoạn dị biệt duy nhất khi người nghe tạm rời xa nhạc nhảy điện tử để tìm về âm hưởng mộc mạc hoài niệm.


In [ ]:
data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
if 'df_country_year' not in locals():
    df_country_year = pd.read_parquet(f'{data_dir}/mart_country_year.parquet')

# Lọc dữ liệu năm 2021 và loại bỏ bảng xếp hạng Global
df_map = df_country_year[(df_country_year['year'] == 2021) & (df_country_year['chart_country'] != 'Global')].dropna(subset=['country_code'])

map_metrics = [
    ('valence_weighted', 'Valence: Trầm buồn sang Tươi vui', 'Tealrose'),
    ('danceability_weighted', 'Danceability: Độ bắt nhịp khiêu vũ', 'Viridis'),
    ('acousticness_weighted', 'Acousticness: Độ mộc mạc nhạc cụ', 'YlOrBr'),
    ('energy_weighted', 'Energy: Mức độ sôi động và cường độ', 'Plasma')
]

fig_map = go.Figure()

for i, (col, label, colorscale) in enumerate(map_metrics):
    fig_map.add_trace(go.Choropleth(
        locations=df_map['country_code'],
        z=df_map[col],
        text=df_map['chart_country'],
        name=label.split(':')[0],
        colorscale=colorscale,
        colorbar_title=col.replace('_weighted', '').capitalize(),
        visible=(i == 0),
        hovertemplate='<b>%{text}</b><br>' + col.replace('_weighted', '').capitalize() + ': %{z:.3f}<extra></extra>'
    ))

buttons_map = []
for i, (col, label, colorscale) in enumerate(map_metrics):
    vis = [False] * len(map_metrics)
    vis[i] = True
    buttons_map.append(dict(
        label=label.split(':')[0].strip(),
        method='update',
        args=[
            {'visible': vis},
            {'title': f'Bản đồ phân bổ âm học toàn cầu: {label}'}
        ]
    ))

fig_map.update_layout(
    title='Bản đồ phân bổ âm học toàn cầu: ' + map_metrics[0][1],
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    updatemenus=[dict(
        buttons=buttons_map,
        direction='down',
        x=0.01,
        y=1.15,
        showactive=True
    )],
    height=600,
    margin=dict(l=0, r=0, t=75, b=20)
)
fig_map.show()
